# Data verification notebook

This notebook is a guardrail before cleaning.

The goal is to validate the real data in PostgreSQL instead of assuming that a generic Airbnb schema or a previous AI note is correct.

We check:
- tables exist and row counts match expectations
- column types and null rates are real
- duplicate keys are identified
- suspicious columns (dates, booleans, prices, IDs) are inspected before we clean them

The previous notes are useful as a checklist, but we still verify each point against the actual database.

In [3]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(Path(r"C:\Users\LENOVO\Desktop\csv-to-postgres") / ".env")

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PGUSER','postgres')}:{os.getenv('PGPASSWORD','')}@{os.getenv('PGHOST','localhost')}:{os.getenv('PGPORT','5432')}/{os.getenv('PGDATABASE','csv_project')}"
)

print("Connected to PostgreSQL database.")

sql = """
SELECT table_name, COUNT(*) AS row_count
FROM (
    SELECT 'listings' AS table_name, id FROM listings
    UNION ALL
    SELECT 'calendar' AS table_name, listing_id FROM calendar
    UNION ALL
    SELECT 'reviews' AS table_name, id FROM reviews
) x
GROUP BY table_name
ORDER BY table_name;
"""

counts = pd.read_sql(sql, engine)
print(counts)

Connected to PostgreSQL database.
  table_name  row_count
0   calendar   28352835
1   listings      77679
2    reviews      29222


In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(Path(r"C:\Users\LENOVO\Desktop\csv-to-postgres") / ".env")
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}@"
    f"{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

csv_path = Path(r"C:\Users\LENOVO\Desktop\csv-to-postgres\data\raw\calendar.csv")
print(f"Loading calendar from: {csv_path}")

chunks = pd.read_csv(csv_path, chunksize=250000)
for i, chunk in enumerate(chunks, start=1):
    chunk.to_sql("calendar", engine, if_exists="replace" if i == 1 else "append", index=False, method="multi")
    print(f"Loaded chunk {i} rows={len(chunk):,}")

print("calendar table refreshed successfully")

# Quick validation after import
with engine.connect() as conn:
    row_count = conn.exec_driver_sql("SELECT COUNT(*) FROM calendar").scalar()
    overlap = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM (SELECT DISTINCT listing_id FROM calendar INTERSECT SELECT DISTINCT id FROM listings) x"
    ).scalar()
    print(f"calendar rows after load: {row_count:,}")
    print(f"listing-calendar overlap: {overlap:,}")


Loading calendar from: C:\Users\LENOVO\Desktop\csv-to-postgres\data\raw\calendar.csv
Loaded chunk 1 rows=250,000
Loaded chunk 2 rows=250,000
Loaded chunk 3 rows=250,000
Loaded chunk 4 rows=250,000
Loaded chunk 5 rows=250,000
Loaded chunk 6 rows=250,000
Loaded chunk 7 rows=250,000
Loaded chunk 8 rows=250,000
Loaded chunk 9 rows=250,000
Loaded chunk 10 rows=250,000


In [ ]:
# Schema inspection for all 3 tables
schema_sql = """
SELECT table_name,
       column_name,
       data_type,
       is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name IN ('listings', 'calendar', 'reviews')
ORDER BY table_name, ordinal_position;
"""

schema = pd.read_sql(schema_sql, engine)
print(schema.head(50))
print(f"\nTotal columns inspected: {len(schema)}")

In [ ]:
# Duplicate key checks
checks = {
    "listings_id_unique": "SELECT COUNT(*) AS duplicates FROM (SELECT id FROM listings GROUP BY id HAVING COUNT(*) > 1) x;",
    "calendar_listing_date_unique": "SELECT COUNT(*) AS duplicates FROM (SELECT listing_id, date FROM calendar GROUP BY listing_id, date HAVING COUNT(*) > 1) x;",
    "reviews_id_unique": "SELECT COUNT(*) AS duplicates FROM (SELECT id FROM reviews GROUP BY id HAVING COUNT(*) > 1) x;"
}

for label, query in checks.items():
    result = pd.read_sql(query, engine)
    print(f"{label}:\n{result.to_string(index=False)}\n")

In [ ]:
# Null audit for the suspicious columns noted in the project brief
suspicious = [
    'neighborhood_overview',
    'host_since',
    'host_response_time',
    'host_thumbnail_url',
    'host_neighbourhood',
    'host_verifications',
    'neighbourhood',
    'instant_bookable',
    'host_profile_id',
    'price',
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'has_availability',
    'last_scraped',
    'calendar_last_scraped',
    'first_review',
    'last_review'
]

existing = pd.read_sql(
    """
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = 'listings'
    """,
    engine
)
existing_cols = set(existing['column_name'])

cols_to_check = [c for c in suspicious if c in existing_cols]

null_sql = "SELECT " + ", ".join(
    f"SUM(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END) AS {c}_nulls" for c in cols_to_check
) + " FROM listings;"

nulls = pd.read_sql(null_sql, engine)
print(nulls)

In [ ]:
# Inspected values for the obvious data-shape problems
preview_sql = """
SELECT
    id,
    host_id,
    host_profile_id,
    price,
    host_is_superhost,
    host_has_profile_pic,
    host_identity_verified,
    has_availability,
    last_scraped,
    calendar_last_scraped,
    first_review,
    last_review
FROM listings
LIMIT 10;
"""

preview = pd.read_sql(preview_sql, engine)
print(preview.to_string(index=False))

In [ ]:
# Calendar validation: date coverage and key fields
calendar_sql = """
SELECT
    COUNT(*) AS rows,
    MIN(date) AS min_date,
    MAX(date) AS max_date,
    COUNT(DISTINCT listing_id) AS distinct_listings,
    COUNT(DISTINCT date) AS distinct_dates,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price_rows,
    SUM(CASE WHEN available IS NULL THEN 1 ELSE 0 END) AS null_available_rows
FROM calendar;
"""

calendar_summary = pd.read_sql(calendar_sql, engine)
print(calendar_summary)

In [ ]:
# Reviews validation: null checks and date-like field review data
reviews_sql = """
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT id) AS distinct_ids,
    COUNT(DISTINCT listing_id) AS distinct_listings,
    MIN(date) AS min_review_date,
    MAX(date) AS max_review_date,
    SUM(CASE WHEN reviewer_id IS NULL THEN 1 ELSE 0 END) AS null_reviewer_id,
    SUM(CASE WHEN comments IS NULL THEN 1 ELSE 0 END) AS null_comments
FROM reviews;
"""

reviews_summary = pd.read_sql(reviews_sql, engine)
print(reviews_summary)

## Interpretation checklist

This notebook is intentionally conservative.

A few points are worth validating before writing the cleaning SQL:
- `host_profile_id` should be treated as suspect because of float precision loss and because `host_id` already provides the stable key.
- `price` is likely text and will need numeric conversion after stripping currency symbols and commas.
- Boolean-like columns may be stored as `'t'/'f'`, not real booleans.
- Date columns may be text and must be verified before conversion.
- `calendar` and `reviews` should stay at their original grain for later cross-filtering in Power BI; we should not pre-aggregate them prematurely.

This is the right place to protect the data model before cleaning.